In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from pandas.tseries.offsets import DateOffset
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
df1 = pd.read_csv('idx_cleaned_data1.csv');
df2 = pd.read_csv('idx_cleaned_data2.csv');
df3 = pd.read_csv('idx_cleaned_data3.csv');
df = pd.concat([df1, df2, df3], ignore_index=True);
df.columns

/tmp/ipython-input-715/3780464941.py:2: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('idx_cleaned_data2.csv');
/tmp/ipython-input-715/3780464941.py:3: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df3 = pd.read_csv('idx_cleaned_data3.csv');


Index(['BuyerAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN', 'CloseDate',
       'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'DaysOnMarket', 'BuyerOfficeName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish',
       'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'City',
       'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate',
       'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince',
       'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'Group'],
      dtype

In [ ]:
df['CloseDate'] = pd.to_datetime(df['CloseDate'], yearfirst=True);

last_date = df['CloseDate'].max();
test_date = last_date-DateOffset(months=1);
train_date = test_date-DateOffset(months=6);

train_df = df[(df['CloseDate'] > train_date) & (df['CloseDate'] < test_date)];
test_df = df[(df['CloseDate'] >= test_date)];
test_df = test_df[(test_df['ClosePrice'] > test_df['ClosePrice'].quantile(0.05)) & (test_df['ClosePrice'] < test_df['ClosePrice'].quantile(0.95))]

train_y = train_df['ClosePrice'];
test_y = test_df['ClosePrice'];

In [ ]:
#ran into some issues with flooring, levels - will properly onehotencode
drop_cols = ['Flooring', 'Levels', 'UnparsedAddress', 'PostalCode', 'City', 'StreetNumberNumeric', 'PropertyType', 'PropertySubType', 'ClosePrice', 'CloseDate', 'DaysOnMarket', 'BuyerOfficeName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'BuyerAgentAOR', 'BuyerOfficeAOR', 'ContractStatusChangeDate', 'PurchaseContractDate', 'MLSAreaMajor', 'CountyOrParish', 'ElementarySchool', 'SubdivisionName', 'ListingContractDate', 'HighSchool', 'HighSchoolDistrict', 'StateOrProvince', 'MiddleOrJuniorSchool'];
train_X = train_df.drop(columns=drop_cols);
test_X = test_df.drop(columns=drop_cols);

In [ ]:
numeric_cols = list(train_X.dtypes[(train_X.dtypes == 'float64') | (train_X.dtypes == 'int')].index);
numeric_cols.remove('Latitude');
numeric_cols.remove('Longitude');
categorical_cols = list(train_X.dtypes[train_X.dtypes == 'object'].index);
ct = ColumnTransformer(
    [("StandardScaler", StandardScaler(), numeric_cols),
     ("OneHotEncode", OneHotEncoder(), categorical_cols)], remainder='passthrough');
train_X = ct.fit_transform(train_X);
test_X = ct.transform(test_X);

In [ ]:
def calculate_mdape(model, test_X, test_y):
    return np.median(np.divide(np.abs(model.predict(test_X)-test_y), test_y))*100;

In [ ]:
olsReg = LinearRegression().fit(train_X, train_y);
print("R^2: "+str(olsReg.score(test_X, test_y)));
print("MdAPE: "+str(calculate_mdape(olsReg, test_X, test_y)));

R^2: -0.16626391080609149
MdAPE: 60.511984433955476


In [ ]:
ridgeReg = Ridge().fit(train_X, train_y);
print("R^2: "+str(ridgeReg.score(test_X, test_y)));
print("MdAPE: "+str(calculate_mdape(ridgeReg, test_X, test_y)));

R^2: -0.16618995216008958
MdAPE: 60.514743298590744


In [ ]:
lassoReg = Lasso().fit(train_X, train_y);
print("R^2: "+str(lassoReg.score(test_X, test_y)));
print("MdAPE: "+str(calculate_mdape(lassoReg, test_X, test_y)));

R^2: -0.16625066939127042
MdAPE: 60.51455327219368


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.522e+17, tolerance: 3.763e+14
  model = cd_fast.enet_coordinate_descent(


In [ ]:
# Train Random Forest Regressor
rfReg = RandomForestRegressor(random_state=42).fit(train_X, train_y);

print("R^2: " + str(rfReg.score(test_X, test_y)));
print("MdAPE: " + str(calculate_mdape(rfReg, test_X, test_y)));

R^2: -3.281198094413468
MdAPE: 10.21875571009699


In [ ]:
# Grid Search
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [10, 20, 30, 40, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Random Forest Regressor
rf = RandomForestRegressor(random_state=42)

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist,
                                   n_iter=50, cv=3, verbose=2, random_state=42,
                                   n_jobs=-1)

# Fit random search model
random_search.fit(train_X, train_y)

print("Best parameters found: ", random_search.best_params_)
print("Best R^2 score found: ", random_search.best_score_)

best_rf_reg = random_search.best_estimator_

Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
69 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
68 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", l

Best parameters found:  {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}
Best R^2 score found:  0.030862009322091155


In [ ]:
# Print best Random Forest Regressor from RandomizedSearchCV
print("R^2 : " + str(best_rf_reg.score(test_X, test_y)))
print("MdAPE : " + str(calculate_mdape(best_rf_reg, test_X, test_y)))

R^2 : 0.3259219185727056
MdAPE : 24.01883774808407
